In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ayushikhandelia/tables/tables.json
/kaggle/input/datasets/ayushikhandelia/spider-dataset-train/train_spider.json


In [4]:
import json

# # with open("/kaggle/input/datasets/ayushikhandelia/spider-dataset/train_others.json") as f:
#     data = json.load(f)

# print(len(data))
# print(data[0])

In [5]:
!pip install -q transformers peft bitsandbytes datasets accelerate sqlparse

In [6]:
import torch
import json
import sqlparse
import sqlite3
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

In [7]:
# model_name = "mistralai/Mistral-7B-v0.1"
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# tokenizer.pad_token = tokenizer.eos_token
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     device_map="auto"
# )
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=32,
#     target_modules=["q_proj", "v_proj"],
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM"
# )
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [ ]:
# import json

# with open("/kaggle/input/spider-dataset/train_spider.json") as f:
#     raw = json.load(f)

In [ ]:
data = []

with open("/kaggle/input/datasets/ayushikhandelia/spider-dataset-train/train_spider.json", "r") as f:
    raw = json.load(f)

In [ ]:
with open("/kaggle/input/datasets/ayushikhandelia/tables/tables.json") as f:
    tables = json.load(f)

In [ ]:
db_schemas = {}

for table in tables:
    db_id = table["db_id"]
    
    table_names = table["table_names_original"]
    column_names = table["column_names_original"]
    
    schema = {}

    for table_id, col_name in column_names:
        if table_id == -1:
            continue
        
        table_name = table_names[table_id]
        
        if table_name not in schema:
            schema[table_name] = []
        
        schema[table_name].append(col_name)

    db_schemas[db_id] = schema

In [ ]:
def build_schema(db):
    schema = "Tables:\n"
    
    for table, columns in db.items():
        schema += f"{table}({', '.join(columns)})\n"
    
    return schema

In [ ]:
clean_data = []

for ex in raw:

    schema_dict = db_schemas[ex["db_id"]]
    schema = build_schema(schema_dict)

    messages = [
        {
            "role": "system",
            "content": "You are an expert SQL assistant. Generate only valid SQL queries."
        },
        {
            "role": "user",
            "content": f"""
Database Schema:
{schema}

Question:
{ex['question']}
"""
        },
        {
            "role": "assistant",
            "content": ex["query"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )

    clean_data.append({"text": text})

In [ ]:
dataset = Dataset.from_list(clean_data)

In [ ]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize)

In [ ]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/sql-genie",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     num_train_epochs=1,
#     learning_rate=2e-4,
#     logging_steps=10,
#     eval_strategy="no",
#     save_strategy="no",
#     fp16=True,
#     report_to="none"
# )
training_args = TrainingArguments(
    output_dir="/kaggle/working/sql-genie",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="no",
    save_strategy="epoch",
    fp16=True,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()

In [ ]:
def is_valid_sql(query):
    try:
        sqlparse.parse(query)
        return True
    except:
        return False


def schema_check(query, schema):
    cols = schema.lower().split()
    for word in query.lower().split():
        if word.isalpha() and word not in cols:
            return False
    return True

In [14]:
def optimize_query(query):
    query = query.strip()

    # 🔹 Remove repeated spaces
    query = " ".join(query.split())

    # 🔹 Remove SELECT *
    if "select *" in query.lower():
        query = query.replace("*", "id")

    # 🔹 Normalize operators
    query = query.replace(" = ", "=")
    query = query.replace(" > ", ">")
    query = query.replace(" < ", "<")

    # 🔹 Remove duplicate WHERE conditions
    if "where" in query.lower():
        parts = query.split("WHERE")
        conditions = parts[1].split("AND")
        conditions = list(set([c.strip() for c in conditions]))
        query = parts[0] + "WHERE " + " AND ".join(conditions)

    return query

In [ ]:
old_query = ""

In [9]:
def generate_sql(schema, question):
#     prompt = f"""You are an expert SQL generator.

# Database Schema:
# {schema}

# Generate a SQL query.

# Question:
# {question}

# Older Queries:
# {old_query}

# SQL Query:"""
    messages = [
        {
            "role": "system",
            "content": "You are an expert SQL assistant."
        },
        {
            "role": "user",
            "content": f"""
    Database Schema:
    {schema}
    
    Question:
    {question}
    
    Generate only the SQL query.
    """
        }
    ]
        # inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
    )
    
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to("cuda")

    # output = model.generate(
    #     **inputs,
    #     max_new_tokens=120,
    #     temperature=0.1,
    #     do_sample=False,
    #     pad_token_id=tokenizer.eos_token_id
    # )

    output = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.2,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1
    )
    sql = tokenizer.decode(output[0], skip_special_tokens=True)

    # 🔹 APPLY OPTIMIZATION
    optimized_sql = optimize_query(sql)

    return optimized_sql

In [10]:
print(model.config._name_or_path)
print(tokenizer.name_or_path)

Qwen/Qwen2.5-1.5B-Instruct
Qwen/Qwen2.5-1.5B-Instruct


In [11]:
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear4bit(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear4bit(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1

In [12]:
import torch

x = torch.tensor([1,2,3]).cuda()

print(x)

tensor([1, 2, 3], device='cuda:0')


In [15]:
schema = """
Tables:
students(id, name, age)
marks(student_id, subject, score)

Foreign Keys:
marks.student_id = students.id
"""

question = "List names of students who scored more than 90 in Math"

sql = generate_sql(schema, question)

print(sql)

system You are an expert SQL assistant. user Database Schema: Tables: students(id, name, age) marks(student_id, subject, score) Foreign Keys: marks.student_id=students.id Question: List names of students who scored more than 90 in Math Generate only the SQL query. assistant ```sql SELECT s.name FROM students s JOIN marks m ON s.id=m.student_id WHERE m.score>90; ``` AND m.subject='Math'


In [ ]:
print("Valid SQL:", is_valid_sql(sql))
if is_valid_sql(sql):
    old_query += sql
print("Schema Valid:", schema_check(sql, schema))

In [2]:
# SAVE LoRA adapter
# model.save_pretrained("/kaggle/working/sql-genie-lora")
# tokenizer.save_pretrained("/kaggle/working/sql-genie-lora")
model.save_pretrained(
    "/kaggle/working/sql-genie-qwen"
)

tokenizer.save_pretrained(
    "/kaggle/working/sql-genie-qwen"
)
print("Saved LoRA adapter!")

NameError: name 'model' is not defined

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    "/kaggle/working/sql-genie-qwen"
)

In [16]:
tokenizer.save_pretrained("/kaggle/working/sql-genie/checkpoint-1314")

('/kaggle/working/sql-genie/checkpoint-1314/tokenizer_config.json',
 '/kaggle/working/sql-genie/checkpoint-1314/chat_template.jinja',
 '/kaggle/working/sql-genie/checkpoint-1314/tokenizer.json')

In [17]:
!pip install gradio -q

import gradio as gr

def generate_sql_ui(schema, question):
    return generate_sql(schema, question)

demo = gr.Interface(
    fn=generate_sql_ui,
    inputs=[
        gr.Textbox(lines=10, label="Database Schema"),
        gr.Textbox(lines=3, label="Question")
    ],
    outputs=gr.Code(language="sql"),
    title="SQL Genie"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://f369fdc732ac39dff0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
